# 00 — Environment check and raw data contract

**Deliverables D0 and D1.**

This notebook does no analysis. It answers one question: *is the environment and the
delivered data exactly what the rest of the pipeline assumes?*

Run it first, and run it again whenever the data delivery changes. It reads Parquet
**footers only** — no row group is decompressed, so the whole notebook completes in a
couple of seconds against 86.6 M rows.

| | |
|---|---|
| **D0** | package skeleton, config and a DuckDB connection with the relations registered |
| **D1** | raw inventory, schema contract, and the alias mapping |

Nothing is written to disk.

## Setup

The bootstrap mirrors the `bms_sa_review` notebooks: find the repository root by walking
upwards, then put it on `sys.path` so both packages import cleanly regardless of where
Jupyter was started.

If your data lives somewhere other than
`~/OneDrive - UNSW/Documents/CICCADA - Data/solar edge`, set the environment variable
`CICCADA_SE_DATA_ROOT` before importing.

In [ ]:
import sys
from pathlib import Path

_current = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_current, *_current.parents)
     if (p / "solar_edge").is_dir() and (p / "bms_sa_review").is_dir()),
    None,
)
if REPO_ROOT is None:
    raise RuntimeError(f"Could not locate the CICCADA repository root from {_current}")
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd

from solar_edge.config import se_config as C
from solar_edge.lib import se_store
from solar_edge.lib import se_diagnostics as diag

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

print("Repository root:", REPO_ROOT)
print("Data root:      ", C.DATA_ROOT)

## 1. Environment

Required packages must be present. Optional ones are informational — `geopandas` and
`shapely` are only needed from **D4**, where postcodes are mapped to ABS POA-2021
polygon centroids and then to BOM grid points.

Paths must resolve. `STORE_DIR` and `ARTEFACT_DIR` are allowed to be missing: the store
is built up deliverable by deliverable and does not exist yet.

In [ ]:
env = diag.environment_report()
display(env)

missing = env.loc[env["required"] & ~env["pass"]]
if len(missing):
    raise RuntimeError(f"Required items missing:\n{missing[['item', 'detail']]}")
print("Environment OK.")

## 2. Conventions

Everything the ingest step will apply to turn SolarEdge telemetry into the CICCADA
convention. These are printed rather than buried in a comment because they are
methodological choices, and from **D5** they become part of `manifest()` so they travel
with every published number.

**The sign convention is the one to read carefully.** SolarEdge uses a *mixed*
convention:

- **Active power** is reported as a production magnitude — over all of 2025 it has
  `min = 0` and no negative values, so it is already generator-positive. No change.
- **Reactive power** is reported in the **load (consumer) convention**, where
  *positive = absorbing*. CICCADA and AS/NZS 4777.2 Fig 3.2 use the **generator
  convention**, where *negative = absorbing*. So `Q` is multiplied by `-1` at ingest.

After that flip, `Q < 0` means absorbing, and the Method A / Method B tests port from
`voltvar_queries.py` unchanged.

In [ ]:
display(C.describe_conventions())

## 3. Connect

An in-memory DuckDB connection. No data is copied into a database file — the store
stays as Parquet on disk, readable by pandas, polars and Arrow, and regenerable from
the raw delivery.

`se_raw` spans all 12 monthly files as a single relation. `se_alias` is the site
mapping CSV. Store tables are registered only once they have been built.

In [ ]:
con = se_store.connect(verbose=True)
display(se_store.relations(con))

## 4. Raw inventory

Footer metadata for each delivered file.

Two columns matter beyond the row counts:

- **`schema_fingerprint`** — a hash of the ordered `(column, type)` list. One distinct
  value across all 12 files means one schema, so a single query can read the lot.
- **`n_row_groups`** — every file has exactly **one** row group holding millions of
  rows. There are therefore no row-group statistics to prune on, and any predicate
  forces a full column scan. This is the specific reason **D3** re-partitions the data
  rather than querying the delivery directly.

In [ ]:
inv = diag.raw_inventory(con)
display(inv[["month", "n_rows", "n_row_groups", "rows_per_row_group",
             "n_columns", "size_mb", "schema_fingerprint"]])

print(f"Files:        {len(inv)}")
print(f"Total rows:   {inv.n_rows.sum():,}")
print(f"Total size:   {inv.size_mb.sum():,.0f} MB compressed on disk")
print(f"Writer:       {inv.created_by.iloc[0]}")
print()
print("For scale: as float64 in pandas this would be roughly "
      f"{inv.n_rows.sum() * 14 * 8 / 1024**3:.1f} GB in memory, "
      "which is why nothing is ever loaded whole.")

## 5. Raw schema

The 15 delivered columns. `se_config.RAW_COLUMNS` is the declared contract; this is what
is actually there.

Note what is **absent**: no nameplate capacity, no inverter model, no DNSP, no install
date, and no irradiance. Site metadata is postcode and state only.

In [ ]:
schema = diag.raw_schema(con, Path(inv.path.iloc[0]))
schema["declared"] = schema.column_name.map(C.RAW_COLUMNS)
display(schema)

## 6. D1 checks

Three groups of assertions:

1. **Schema contract** — all files share one schema, and it is the declared one.
2. **Inventory** — 12 months with no gaps, and per-file row counts matching
   `se_config.EXPECTED_RAW_ROWS` (measured 12 Aug 2026).
3. **Alias mapping** — complete, unique, and every state has a timezone mapped.

If the row-count checks fail you have been given a different extract, and the measured
figures in the architecture proposal need re-deriving before anything else runs.

In [ ]:
checks = diag.run_d1_checks(con, inventory=inv)
display(checks)

ok = diag.summarise(checks, label="D1")
assert ok, "D1 checks failed — see above before proceeding to D2."

## 7. Site mapping

The complete site metadata: 1,602 aliases, each with a postcode and a state.

The postcode count is worth noting. The 1,602 sites resolve to a few hundred distinct
postcodes, so the BOM irradiance extract in **D12a** — which is keyed on the nearest BOM
grid point, not on the site — will be far smaller than the site count suggests.

In [ ]:
display(diag.alias_mapping_summary(con))

postcodes = se_store.q(con, "SELECT count(DISTINCT zip_code) AS n_postcodes FROM se_alias")
print(f"Distinct postcodes across the fleet: {int(postcodes.n_postcodes.iloc[0])}")
print("These collapse further into BOM grid points at D4, which sets the D12a extract size.")

## 8. Store status

What has been built so far. Everything is expected to be missing at this point — the
store is populated from **D3** onward.

This table is the SolarEdge analogue of `conformance_queries.table_provenance()`: it
makes it impossible to run an analysis against a store you believed was complete but
was not.

In [ ]:
display(se_store.store_status(con)[["logical_name", "exists", "kind", "size_mb", "n_rows"]])

## What this establishes

- The environment imports cleanly and every required path resolves.
- The delivery is **12 monthly files, 2025-01 to 2025-12, 86,643,185 rows,
  1.6 GB compressed**, with **one schema** across all files.
- The site mapping is **1,602 sites** across NSW, SA and QLD, complete and unique.
- Every file has a **single row group**, which is the performance case for D3.

## Next: D2 — timestamp and DST resolution

The `timestamp` column is a naive string in each site's local civil time, *including*
daylight saving. D2 builds and unit-tests the conversion to `ts_utc` before any data is
rewritten, because a DST error at ingest would silently duplicate an hour each April and
delete one each October.

The acceptance test: recomputed power-weighted diurnal centroids must reproduce
QLD 11.79 / 11.96 h, NSW 11.91 / 13.07 h and SA 12.30 / 13.48 h for June and January.